# Reality2Unity — Week 4 (Colab)

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

repo_dir = Path("/content/Reality2Unity")
repo_url = "https://github.com/mir0173/Reality2Unity.git"

if (repo_dir / ".git").is_dir():
    subprocess.run(["git", "switch", "develop"], cwd=repo_dir, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", "develop"], cwd=repo_dir, check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", "develop", "--single-branch", repo_url, str(repo_dir)],
        check=True,
    )

cli_path = repo_dir / "python" / "reality2unity" / "cli.py"
if not cli_path.is_file():
    raise FileNotFoundError("GitHub develop에 python/reality2unity/cli.py가 있는지 확인하세요.")

In [ ]:
multiview_dir = Path("/content/drive/MyDrive/Reality2Unity/multiview")
image_dir = multiview_dir / "images"

if not image_dir.is_dir():
    raise FileNotFoundError(f"Drive에서 사진 폴더를 찾을 수 없습니다: {image_dir}")

images = sorted(
    path.name
    for path in image_dir.iterdir()
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

if len(images) != 23:
    raise ValueError(f"Terrace 사진 23장을 예상했는데 {len(images)}장을 찾았습니다.")

config_path = Path("/content/terrace_drive.json")
config_path.write_text(
    json.dumps({"scene": "terrace", "images": images}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

subprocess.run(
    [
        sys.executable, "-m", "reality2unity.cli", "prepare",
        "--config", str(config_path),
        "--image-dir", str(image_dir),
        "--runs-dir", str(multiview_dir / "runs"),
    ],
    cwd=repo_dir / "python",
    check=True,
)